# AMASS as 3D COCO keypoints — the third arm

Same AMASS takes as `preprocess_amass_smpl.ipynb`, read from `AMASS_COCO3D`: 18 keypoints (COCO-17
plus neck) per frame, no joint model at all. Per joint: body-frame position plus its first two
frame differences, 18 joints x 9 channels.

Kept apart by key, as the SMPL arm is: `source_dataset="AMASS-COCO"`, its own representation
name, its own split sets, its own stats. AMASS_COCO3D covers fewer subsets than the release, so
the three arms are paired into **new** split sets (`pretrain_paired_coco_v1`, `babel_paired_coco`)
rather than shrinking `pretrain_paired_v1` / `babel_paired` under the existing runs.

In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import polars as pl
import yaml

from sometria.babel import import_babel_annotations, import_babel_splits
from sometria.catalog import MotionViewSpec, build_motion_view, create_paired_split, load_splits, sample_key
from sometria.normalization import save_feature_normalization
from sometria.preprocess import ImportConfig, import_coco_dataset
from sometria.representation import build_representation

In [ ]:
RAW_COCO = Path("../data/raw/AMASS_COCO3D")
RAW_BABEL = Path("/home/fziche/nas/MAEVE/dataset/babel")
OUT = Path("../data/processed")

ARMS = ("AMASS", "AMASS-SMPL", "AMASS-COCO")
PRETRAIN_SPLIT = "pretrain_paired_coco_v1"
BABEL_SPLIT = "babel_paired_coco"
NORMALIZATION_NAME = f"{PRETRAIN_SPLIT}_train_clean"
WINDOW_FRAMES = 240

COCO_HUMAN = yaml.safe_load(Path("../config/coco.yaml").read_text())
REPRESENTATIONS = {
    "AMASS":      build_representation("../config/human.yaml"),
    "AMASS-SMPL": build_representation("../config/smpl.yaml"),
    "AMASS-COCO": build_representation("../config/coco.yaml"),
}
for arm, representation in REPRESENTATIONS.items():
    print(f"{arm:11s} {representation}")

AMASS       Representation('opensim_sincos_log_vel_acc_tau_v2', 43 dofs, 5 channels)
AMASS-SMPL  Representation('smpl_rot6d_log_vel_acc_v1', 21 dofs, 18 channels)
AMASS-COCO  Representation('coco18_canonpos_vel_acc_v1', 18 dofs, 9 channels)


## Import

BABEL importers are re-run afterwards so the new rows land on the official splits and labels
(they match on the normalized source path, which `CMU/01/01_01_stageii.csv` shares with the
`.npz` and the OpenSim CSV).

In [3]:
catalog = import_coco_dataset(
    config=ImportConfig(source_dataset="AMASS-COCO", input_root=RAW_COCO, output_root=OUT, pattern="**/*.csv"),
    human=COCO_HUMAN,
    representation=REPRESENTATIONS["AMASS-COCO"],
)
import_babel_splits(babel_root=RAW_BABEL, output_root=OUT)
import_babel_annotations(babel_root=RAW_BABEL, output_root=OUT)

catalog.filter(pl.col("source_dataset") == "AMASS-COCO").group_by("source_subset").len().sort("len", descending=True)

  2%|▏         | 127/6798 [11:10<9:46:42,  5.28s/it] 


KeyboardInterrupt: 

## Three-way paired splits

In [ ]:
create_paired_split(
    output_root=OUT, split_set=PRETRAIN_SPLIT, source_split_set="pretrain_v1",
    arms=ARMS, splits=("train",), min_frames=WINDOW_FRAMES,
)
create_paired_split(
    output_root=OUT, split_set=BABEL_SPLIT, source_split_set="babel_official",
    arms=ARMS, splits=("train", "val", "test"), min_frames=WINDOW_FRAMES,
)
print(
    load_splits(OUT)
    .filter(pl.col("split_set").is_in([PRETRAIN_SPLIT, BABEL_SPLIT]))
    .with_columns(pl.col("sample_id").str.split(":").list.first().alias("arm"))
    .group_by("split_set", "split", "arm").len().sort("split_set", "split", "arm")
)

## Normalization, and the arms hold the same takes

Stats for all three arms on the new split, so the OpenSim and SMPL arms can be retrained on
exactly the COCO arm's takes (`dataloader.normalization` + `split_set` overrides).

In [ ]:
def pretrain_spec(arm: str) -> MotionViewSpec:
    return MotionViewSpec(
        representation=REPRESENTATIONS[arm].name, split_set=PRETRAIN_SPLIT, split="train",
        source_datasets=(arm,), min_frames=WINDOW_FRAMES,
    )


views = {arm: build_motion_view(OUT, pretrain_spec(arm)) for arm in ARMS}
takes = {arm: set(v["source_path"].map_elements(sample_key, return_dtype=pl.String)) for arm, v in views.items()}
assert takes["AMASS"] == takes["AMASS-SMPL"] == takes["AMASS-COCO"], "arms disagree on takes"

for arm, representation in REPRESENTATIONS.items():
    path = save_feature_normalization(
        output_root=OUT, samples=views[arm], name=NORMALIZATION_NAME,
        representation=representation, spec=pretrain_spec(arm),
    )
    print(f"{arm:11s} {len(views[arm]):>6,} takes  {views[arm]['duration'].sum() / 3600:>7.1f} h  ->  {path}")

## Training

```
uv run python train.py --config config/pretrain_mae.yaml config/dataloader/amass_coco.yaml config/mae/medium.yaml --output runs/pretrain/amass_coco/medium_100ep
```